# Dynamic Programming (DP) and Adaptive Dynamic Programming (ADP) — From Scratch

---

## 1. What is Dynamic Programming (DP)?

**Dynamic programming (DP)** is a general method for solving **multi-step decision problems** by breaking them into **overlapping subproblems** and using a **recursive relationship** (the **Bellman equation**) to compute optimal solutions efficiently.

Two key ideas:
1. **Optimal substructure**: an optimal solution can be built from optimal solutions to smaller subproblems.
2. **Overlapping subproblems**: the same subproblems occur repeatedly, so we store their solutions (**memoization / tabulation**).

DP appears in:
- shortest paths, knapsack, edit distance
- scheduling, planning, inventory control
- **control and reinforcement learning (MDPs)**: DP = planning when the model is known

---

## 2. The Canonical DP Setup

A DP problem usually includes:

- **State** $s$: sufficient information to make the next decision.
- **Action** $a$: a choice available in state $s$.
- **Transition**:
  - deterministic: $s' = f(s,a,w)$ (with noise $w$)
  - stochastic: $P(s' \mid s,a)$
- **Reward / cost**:
  - reward: $r(s,a)$
  - cost: $c(s,a)$
- **Objective**:
  - maximize total expected reward, or minimize total expected cost, across time.

---

## 3. Two Main DP Families

### 3.1 Finite-horizon DP

Time steps: $t = 0,1,\dots,T$.

Define the value function:
- $V_t(s)$ = maximum expected total reward from time $t$ onward when in state $s$.

Bellman recursion:
$$
V_t(s) = \max_{a \in \mathcal{A}(s)}
\left(
r_t(s,a) + \mathbb{E}\left[V_{t+1}(S_{t+1}) \mid S_t=s, A_t=a\right]
\right)
$$

Boundary condition:
$$
V_T(s) = \text{terminal payoff}
$$

This is computed by **backward induction**:
$$
V_T \rightarrow V_{T-1} \rightarrow \dots \rightarrow V_0
$$

---

### 3.2 Infinite-horizon DP (discounted)

Discount factor: $\gamma \in (0,1)$.

Value function:
$$
V(s) = \max_a
\left(
r(s,a) + \gamma \, \mathbb{E}\left[V(S') \mid S=s, A=a\right]
\right)
$$

---

## 4. Markov Decision Processes (MDPs): DP’s Core Framework

An MDP is:
$$
(\mathcal{S}, \mathcal{A}, P, r, \gamma)
$$

- $\mathcal{S}$: state space  
- $\mathcal{A}$: action space  
- $P(s' \mid s,a)$: transition dynamics  
- $r(s,a)$: reward  
- $\gamma$: discount factor  

### 4.1 Policies

A policy $\pi$ determines actions:
- stochastic: $\pi(a \mid s)$
- deterministic: $a = \pi(s)$

### 4.2 Value functions under a policy

State value:
$$
V^\pi(s) = \mathbb{E}_\pi\left[\sum_{t=0}^\infty \gamma^t r(S_t, A_t) \mid S_0=s\right]
$$

Action value:
$$
Q^\pi(s,a) = \mathbb{E}_\pi\left[\sum_{t=0}^\infty \gamma^t r(S_t, A_t) \mid S_0=s, A_0=a\right]
$$

---

## 5. Bellman Equations

### 5.1 Bellman Expectation Equation (policy evaluation)

For a fixed policy $\pi$:
$$
V^\pi(s) =
\sum_a \pi(a \mid s)
\left[
r(s,a) + \gamma \sum_{s'} P(s' \mid s,a) V^\pi(s')
\right]
$$

### 5.2 Bellman Optimality Equation (optimal control)

Optimal state value:
$$
V^*(s) =
\max_a
\left[
r(s,a) + \gamma \sum_{s'} P(s' \mid s,a) V^*(s')
\right]
$$

Optimal action value:
$$
Q^*(s,a) =
r(s,a) + \gamma \sum_{s'} P(s' \mid s,a)\max_{a'} Q^*(s',a')
$$

Optimal policy is greedy:
$$
\pi^*(s) \in \arg\max_a Q^*(s,a)
$$

---

## 6. Algorithms for Solving DP

### 6.1 Policy Evaluation (iterative)

Given $\pi$, start with $V_0$ and iterate:
$$
V_{k+1}(s) \leftarrow
\sum_a \pi(a \mid s)
\left[
r(s,a) + \gamma \sum_{s'}P(s' \mid s,a)V_k(s')
\right]
$$

### 6.2 Policy Improvement

Improve $\pi$ using the evaluated value function:
$$
\pi_{\text{new}}(s)
\in \arg\max_a
\left[
r(s,a) + \gamma \sum_{s'}P(s' \mid s,a)V^\pi(s')
\right]
$$

### 6.3 Policy Iteration

Repeat:
1. policy evaluation
2. policy improvement  
until policy stops changing.

---

### 6.4 Value Iteration

Use optimality backups directly:
$$
V_{k+1}(s) \leftarrow
\max_a
\left[
r(s,a) + \gamma \sum_{s'} P(s' \mid s,a) V_k(s')
\right]
$$

---

## 7. DP vs Recursion vs Greedy

- **Recursion**: recomputes subproblems repeatedly $\Rightarrow$ often exponential time
- **DP**: caches results $\Rightarrow$ efficient
- **Greedy**: locally best choice; may fail globally
- **DP**: explicitly accounts for long-term consequences via $V(s)$

---

## 8. The Curse of Dimensionality

Exact DP becomes infeasible when:
- $|\mathcal{S}|$ is huge or continuous
- $|\mathcal{A}|$ is huge or continuous
- transitions $P(s' \mid s,a)$ are unknown or expensive

Tabular DP often costs about:
$$
O(|\mathcal{S}| \cdot |\mathcal{A}| \cdot |\mathcal{S}|)
$$
per full sweep.

This motivates **Approximate DP / Adaptive DP**.

---

# 9. Adaptive Dynamic Programming (ADP): From Scratch

People use ADP in two closely related senses:

1. **Approximate Dynamic Programming**: use approximations to scale DP.
2. **Adaptive Dynamic Programming**: learn the model and/or value function when unknown or changing.

In practice, ADP means:
> **DP ideas + approximation + learning from data**

ADP strongly overlaps with **reinforcement learning (RL)**.

---

## 10. The Core ADP Idea

DP requires either:
- knowing the model $P(s' \mid s,a)$ and $r(s,a)$, or
- being able to sample transitions from experience.

ADP uses:
- a learned value approximation $\hat V(s;w)$ or $\hat Q(s,a;w)$,
- and/or a learned model $\hat P, \hat r$,

and then performs Bellman-like updates approximately.

---

## 11. Three Major ADP Approaches

---

## 11.1 Value Function Approximation (often model-free)

Instead of storing $V(s)$ for all states, we approximate:
$$
V(s) \approx \hat V(s;w)
$$
or
$$
Q(s,a) \approx \hat Q(s,a;w)
$$

### Fitted Value Iteration / Fitted Q Iteration

Given transitions $(s,a,r,s')$, form targets:
$$
y = r + \gamma \max_{a'} \hat Q(s',a';w)
$$
and fit $\hat Q(s,a;w)$ to match $y$ via regression.

---

## 11.2 Temporal Difference (TD) Learning (policy evaluation from data)

For a fixed policy, TD(0) defines the TD error:
$$
\delta_t = r_t + \gamma \hat V(S_{t+1};w) - \hat V(S_t;w)
$$

Update parameters $w$ to reduce $\delta_t$ (e.g., stochastic gradient descent).

---

## 11.3 Q-learning (control from data, off-policy)

Tabular Q-learning update:
$$
Q(s,a) \leftarrow Q(s,a) +
\alpha\left(
r + \gamma \max_{a'}Q(s',a') - Q(s,a)
\right)
$$

This is essentially **adaptive value iteration** without knowing $P$.

---

## 11.4 Policy Approximation (Actor-Critic)

Represent policy directly:
- stochastic: $\pi(a \mid s;\theta)$
- deterministic: $a = \mu(s;\theta)$

Actor-critic:
- **critic** learns $\hat V$ or $\hat Q$
- **actor** updates policy parameters $\theta$ using the critic

---

## 11.5 Model-based ADP

Learn a model:
$$
\hat P(s' \mid s,a), \quad \hat r(s,a)
$$

Then plan with DP-like methods on the learned model.

This can be sample-efficient but can fail if model errors compound.

---

# 12. Key Concepts for ADP

---

## 12.1 Bellman Error / Residual

The optimal Bellman equation is:
$$
V(s) = \max_a \left(r(s,a) + \gamma \mathbb{E}[V(S')]\right)
$$

The **Bellman residual** measures mismatch between left and right sides.

ADP tries to reduce this mismatch approximately.

---

## 12.2 Bootstrapping

Using current estimates to define targets like:
$$
r + \gamma \hat V(s')
$$
is called **bootstrapping**.

Pros:
- efficient learning
- online updates possible

Cons:
- can become unstable with function approximation and off-policy learning

This instability is sometimes called the **deadly triad**:
- function approximation
- bootstrapping
- off-policy learning

---

## 12.3 Exploration vs Exploitation

Need exploration to learn:
- $\epsilon$-greedy
- softmax / Boltzmann exploration
- optimism bonuses, UCB-style methods

---

## 12.4 On-policy vs Off-policy

- **On-policy**: learn the value of the policy you execute (e.g., SARSA)
- **Off-policy**: learn a target policy while behaving differently (e.g., Q-learning)

---

# 13. DP vs ADP vs RL (Mental Map)

- **DP**: model known, compute exact optimal solutions (planning).
- **ADP**: approximate DP when exact computation is impossible.
- **RL**: learning to act from data; many RL methods are ADP methods.

---

# 14. Tiny Example: Inventory Control (conceptual)

State:
$$
s = \text{inventory level}
$$

Action:
$$
a = \text{order quantity}
$$

Reward:
$$
r(s,a) = \text{sales profit} - \text{holding cost} - \text{stockout cost}
$$

Transition depends on random demand.

- **DP**: if you know demand distribution, compute expectations exactly and run value iteration / policy iteration.
- **ADP**: if you don’t know demand distribution or the state space is huge, learn $\hat V(s)$ from samples and act greedily with respect to it.

---

# 15. When to Use Which

## Use DP when:
- state/action spaces are small enough
- model $P$ is known
- you want exact optimality

## Use ADP when:
- large or continuous state spaces
- unknown or changing dynamics
- you need approximate solutions

---

# 16. Common Pitfalls

- **Bad state definition**: if state doesn’t capture needed history, the Markov property fails.
- **Approximation instability**: bootstrapping + off-policy + function approximation can diverge.
- **Distribution shift**: you learn values on states you don’t visit often.
- **Partial observability**: you see observations $o_t$ instead of states $s_t$; need memory (RNNs) or POMDP methods.

---

# 17. Recommended Study Path

1. Deterministic finite-horizon DP (backward induction)
2. Stochastic finite-horizon DP
3. Infinite-horizon discounted MDPs
4. Value iteration and policy iteration
5. TD learning (policy evaluation from data)
6. Q-learning (control from data)
7. Function approximation (linear, then neural)
8. Actor-critic methods
9. Model-based RL / MPC